In [2]:
# import libraries
from pydantic import BaseModel
from openai import OpenAI
from openai import AsyncOpenAI
import asyncio
import itertools
import math
import random
from dotenv import load_dotenv
import os
import sys
from pathlib import Path
import json
import re
import sys

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import evaluate_seqeval, extract_spans, mention_level_evaluation

In [3]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

def add_special_characters(original_sentence, llm_list):

    matches = []

    # reorder the entities list according to length of the entity
    llm_list = sorted(llm_list, key=len, reverse=True)


    for entity in llm_list:
        # Full-word match but allow possessive 's or ’s
        pattern = r"(?i)(?<![A-Za-z])(" + re.escape(entity) + r")(?:['’]s)?(?![A-Za-z])"

        for match in re.finditer(pattern, original_sentence):
            matches.append(match.span(1))

    
    non_overlapping = []
    occupied = [False] * len(original_sentence)

    for start, end in matches:
        if not any(occupied[start:end]):
            non_overlapping.append((start, end))
            for i in range(start, end):
                occupied[i] = True 

    non_overlapping.sort(key=lambda x: x[0])

    result = original_sentence
    for start, end in reversed(non_overlapping):
        result = result[:start] + "@@" + result[start:end] + "##" + result[end:]

    return result

In [4]:
# initialize empty dataset list
training_data = []
validation_data = []

with open("../../../01_data/training_validation_set/corrected_version/training_set.json", "r") as f:
    raw_training_data = json.load(f)

with open("../../../01_data/training_validation_set/corrected_version/validation_set.json", "r") as f:
    raw_val_data = json.load(f)

# loop through all sentences in the data
for task in raw_training_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    training_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

# loop through all sentences in the data
for task in raw_val_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    validation_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [5]:
# do an estimation of the costs for running inference with different models
model_costs = {
    "4o-mini":
    {"standard": {
        "input": 0.15,
        "output": 0.6
    },
    "batch": {
        "input": 0.075,
        "output": 0.3
    }},
    "4o":
     {"standard": {
        "input": 2.5,
        "output": 10
    },
    "batch": {
        "input": 1.25,
        "output": 5
    }},
    "5-nano": 
    {"standard": {
        "input": 0.05,
        "output": 0.4
    },
    "batch": {
        "input": 0.025,
        "output": 0.2
    }
}}

input_lengths = [1500, 2000, 3000]
token_multiple = 1.3
test_type = "Hyperparameter_Tuning"
gpt_mode = "standard"

print(f"{test_type} Phase with {gpt_mode} processing")
print("-"*70)
for model in model_costs:
    for input in input_lengths:
        if test_type == "Validation":
            input_costs = (input*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "Hyperparameter_Tuning":
            input_costs = (27*input*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (27*100*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "CV":
            input_costs = (input*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "Inference":
            input_costs = (input*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["output"]
        total_costs = input_costs + output_costs
        print(f"Total costs for {model} with input prompt of length {input}: {total_costs:.4f}")
    print("-"*70)

Hyperparameter_Tuning Phase with standard processing
----------------------------------------------------------------------
Total costs for 4o-mini with input prompt of length 1500: 10.0035
Total costs for 4o-mini with input prompt of length 2000: 12.6360
Total costs for 4o-mini with input prompt of length 3000: 17.9010
----------------------------------------------------------------------
Total costs for 4o with input prompt of length 1500: 166.7250
Total costs for 4o with input prompt of length 2000: 210.6000
Total costs for 4o with input prompt of length 3000: 298.3500
----------------------------------------------------------------------
Total costs for 5-nano with input prompt of length 1500: 4.0365
Total costs for 5-nano with input prompt of length 2000: 4.9140
Total costs for 5-nano with input prompt of length 3000: 6.6690
----------------------------------------------------------------------


In [12]:
medium = """
## Task Objective
Extract all social groups from the sentence as a JSON object. Only extract mentions that clearly qualify as social groups. Do not paraphrase the mentions at all.
Include each social group as many times as it occurs in the sentence. If no social groups are present, return an empty JSON array.

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes** (age, ethnicity, income, occupation etc.).

**Do not** extract:
- institutional groups, political groups and state authorities ("small businesses", "armed forces", "Government", "Labour party")
- individual persons or highly specific collectives ("family of a colleague")

**Do** extract:
- subgroups of institutions with shared socio-demographic traits ("business people", "police officers")
- singular forms **only** if it is a generalization to a broader group ("every woman")
- social group component within a composite term ("two-child-limit" -> "child") or title ("Society for Disabled People" -> "Disabled People")
- additional sociodemographic description of the group ("women with mental health problems")
- extract as **one span** if several groups are mentioned and their meaning cannot be understood separately ("veterans, their wifes and children")
- general terms ("communities", "people") **only** if particular group is specified ("local communities", "people with special needs")

## Positive Examples
"Families", "constituents", "victims", "offenders", "consumers"

## Negative Examples
"business", "EU", "Ministers", "armed forces", "Tories"
"""

short = """
## Task Objective
Extract all qualifying social groups from the sentence as a JSON object. Do not paraphrase the mentions at all! If no social groups are present, return an empty JSON array.

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes** (age, ethnicity, income, occupation etc.).

**Do not extract** institutional groups, firms, state authorities, groups based on political opinion or very specific small collectives such as one single family.
**Do extract** groupings of people within an institution and singular forms if it is generalizing to a broader collective.
Extract only the social group component unless there is an additional description to it. 

## Positive Examples
"Families", "constituents", "victims", "offenders", "consumers"

## Negative Examples
"business", "EU", "Ministers", "armed forces", "Tories"
"""

In [6]:
def compile_prompt_summarization(prompt_to_summarize):
    chat = [
        {
            "role": "system",
            "content": """
            Rewrite the system prompt below into a much shorter, clearer system prompt suitable for directly instructing a GPT-4o-mini model in a Named Entity Recognition task.
            The prompt should preserve the most important definitional rules and details while getting rid of unnecessarily complex language or very specific instructions.
            The goal is to enable maximum performance on the mini model.
            Output only the rewritten prompt."""
        },
        {
            "role": "user",
            "content": prompt_to_summarize
        }
    ]
    return chat

prompt = compile_prompt_summarization(medium)
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.chat.completions.create(
    model="gpt-4o",
    messages=prompt
)
synthetically_distilled_prompt = response.choices[0].message.content.strip()

In [11]:
# compile manual few-shot examples in json format
positive_examples = [
    {"text": "Our party would like to encourage all the employers in Leicester to hire those with physical disabilities.",
     "llm_text": '{"social_groups": ["all the employers in Leicester", "those with physical disabilities"]}'},
    {"text": "We want to support every child in this country increasing funding for the UK children's charity. ",
     "llm_text": '{"social_groups": ["every child", "children"]}'},
     {"text": "I want to express my support to all families who struggle because of the economic crisis.",
     "llm_text": '{"social_groups": ["all families who struggle because of the economic crisis"]}'}
]

negative_examples = [
    {"text": "The Government is working in cooperaton with the Labour party on this matter.",
     "llm_text": '{"social_groups": []}'},
    {"text": "Businesses such as small- and medium-sized firms are vital for the economy in this country.",
     "llm_text": '{"social_groups": []}'},
     {"text": "Investment in the armed forces was certainly neglected under the previous Government.",
     "llm_text": '{"social_groups": []}'}
]

In [13]:
# create prompt template for the entity recognition task
def compile_prompt_ner(system_prompt, positive_examples, negative_examples, test_sentence, num_few_shot=4):

        chat = [
                {
                        "role": "system",
                        "content": system_prompt
                }
        ]
        

        num_few_shot_each = num_few_shot//2

        for i in range(num_few_shot_each):

                # add a positive example
                chat.append({"role": "user", "content": f"Sentence: {positive_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": positive_examples[i]["llm_text"]})

                # add a negative example
                chat.append({"role": "user", "content": f"Sentence: {negative_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": negative_examples[i]["llm_text"]})
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

        return chat

In [14]:
# create dictionary storing rate limits
model_limits = {
    "gpt-4o-mini":
    {"token_limit": 200000,
     "request_limit": 500
    },
    "gpt-4o":
    {"token_limit": 30000,
     "request_limit": 500
    },
    "gpt-5-nano":
    {"token_limit": 200000,
     "request_limit": 500
    }
    }


# function to send out the request
async def send_request(client, model_name, prompt, output_class, temp=None, reasoning_effort=None):

    if model_name == "gpt-5-nano" :                                   
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                reasoning = {"effort": reasoning_effort}
                                                )
    elif model_name == "gpt-4o-mini":
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                temperature=temp
                                                )

    response_list = response.output_parsed.social_group
    try:
        if not isinstance(response_list, list):
            return []
        if response_list == ['']:
            return []
        return response_list
    except Exception:
        return []
                                            
                                        
async def dispatch_all(client, model_name, system_message, few_shot_examples, output_class, validation_data, temp, reasoning_effort, safe_interval):
    tasks = []
    for row in validation_data:
        sentence = row["text"]
        prompt = compile_prompt_ner(
            system_message,
            positive_examples,
            negative_examples,
            sentence,
            num_few_shot=few_shot_examples
        )
        # create a task and fire it, do not wait
        task = asyncio.create_task(send_request(client, model_name, prompt, output_class, temp, reasoning_effort))
        tasks.append(task)
        
        # wait before starting the next request
        await asyncio.sleep(safe_interval)

    # gather all results once everything is started
    return await asyncio.gather(*tasks)

In [32]:
# select the model, system message and number of few shot examples
model_name = "gpt-4o-mini"
system_message = medium
few_shot_examples = 4

# set temperature and reasoning effort
temp = 0
reasoning_effort = "medium"

# empirically test a safe rate per minute
if model_name == "gpt-4o":
    safe_rpm = 50
else:
    safe_rpm = 100

# calculate a safe interval in which requests are sent
safe_interval = 60.0 / safe_rpm

# define class for the output
class SocialGroupJSON(BaseModel):
    social_group: list[str]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# get random indices
#random.seed(0)
#random_indices = random.sample(range(1000), 100)
#val_subset = [validation_data[i] for i in random_indices]
val_subset = validation_data
llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples, SocialGroupJSON, val_subset, temp, reasoning_effort, safe_interval)

output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(val_subset, llm_output)]
output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]

In [33]:
# conduct manual error analysis
for idx in range(0, 100):
    print(val_subset[idx]["llm_text"])
    print(output_texts_special_characters[idx])
    print(llm_output[idx])
    print("-"*100)

Its economic plan highlighted the importance of signage in boosting business and tourism on South Hayling Island.
Its economic plan highlighted the importance of signage in boosting business and tourism on South Hayling Island.
[]
----------------------------------------------------------------------------------------------------
Will not this arrangement protect the incomes of @@lower paid barristers##?
Will not this arrangement protect the incomes of @@lower paid barristers##?
['lower paid barristers']
----------------------------------------------------------------------------------------------------
We have dealt with-and continue to deal with-abuse in the @@student## visa system, which was allowed to increase significantly under the previous Labour Government, and non-EU migration is now at the levels of the late 1990s.
We have dealt with-and continue to deal with-abuse in the student visa system, which was allowed to increase significantly under the previous Labour Government, an

In [34]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in val_subset]
pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]

# evaluate at the entity level with seqeval
seqeval_results = evaluate_seqeval(ground_truth_bio, pred_bio)
print(f"Seqeval: {seqeval_results}")

# compute cross-span evaluation
all_true_spans = []
all_predicted_spans = []

for idx in range(len(ground_truth_bio)):
    # get the spans
    all_true_spans.append(extract_spans(ground_truth_bio[idx]))
    all_predicted_spans.append(extract_spans(pred_bio[idx]))

# apply cross-span evaluation
cross_span_results = mention_level_evaluation(all_true_spans, all_predicted_spans)
print(f"Cross-Span: {cross_span_results}")

Seqeval: {'precision': 0.5512820512820513, 'recall': 0.47645429362880887, 'f1': 0.5111441307578009}
Cross-Span: {'precision': 0.5362803281989329, 'recall': 0.512184670354805, 'f1': 0.498317186789179}


In [35]:
# export the preliminary results
gpt_results["gpt-4o-mini"] = {
    "seqeval": seqeval_results,
    "cross_span": cross_span_results
}

with open("../eval_results/evaluation_metrics_gpt.json", "w") as f:
    json.dump(gpt_results, f)

In [23]:
# set up temperatures and top
temperatures = [0, 0.25, 0.5]
reasoning_efforts = [None, "low", "medium", "high"]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# define class for the output
class SocialGroupJSON(BaseModel):
    social_group: list[str]

# tune temperature value for 4o-mini model
model_name = "gpt-4o-mini"
system_message = short
few_shot_examples = 4
safe_rpm = 100
safe_interval = 60.0 / safe_rpm
results_4o_mini = {}
for temp in temperatures:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples, SocialGroupJSON, validation_data, temp, None, safe_interval)
    output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(val_subset, llm_output)]
    output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]
    ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in val_subset]
    pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]
    report = evaluate_seqeval(ground_truth_bio, pred_bio)
    print(f"Report for temperature {temp}: {report}")
    results_4o_mini[temp: report]

# test all reasoning effort values for 5-nano
model_name = "gpt-5-nano"
system_message = medium
few_shot_examples = 4
safe_rpm = 100
safe_interval = 60.0 / safe_rpm
results_5_nano = {}
for re in reasoning_efforts:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples, SocialGroupJSON, validation_data, None, re, safe_interval)
    output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(val_subset, llm_output)]
    output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]
    ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in val_subset]
    pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]
    report = evaluate_seqeval(ground_truth_bio, pred_bio)
    print(f"Report for reasoning effort {re}: {report}")
    results_5_nano[re: report]